USO COMPLETIONS/CHAT PER GENERAZIONE TESTO PERSONALIZZATA

Vediamo come far parlare i modelli direttamente come vogliamo noi.
Modelli capaci di interpretare qualsiasi ruolo, dal medico al pirata. ma soffre di amnesia, dimentica tutto ogni volta che chiude il sipario.
Impareremo a gestire questa amnesia tramite la memoria di contesto.

- Gestione delle cronologia: l'array dei messaggi come memoria di lavoro
- Role-play: definire l'identità tra System, User, e Assistant.
- Deterministico: Controllo della varianza tramite Seed e Temperature, per far si che l'ai non risponda a caso ma in modo riproducibile

Come prima cosa dobbiamo cambiare tipo di modello rispetto alla lezione precedente. 
In precedenza abbiamo usato EmbeddingGemma per trasformare il testo in vettori. 
Qui serve un modello generativo chat/instruct
In LM Studio le interfaccie corrispondenti sono:
- /v1/embeddings: per la lezione precedente sui embeddings
- /v1/completions (completions=completamento di testo)
- v1/chat/completions (completions=completamento di testo)
con completions il modello genera ciò che potrebbe seguire ad una frase data
con la chat, organizzi il contenuto in messaggi con ruoli distinti.
Il risultato rimane sempre testo generato (come in completions). Cambia il modo in cui fornisci istruzioni e contesto.
LM Studio converte automaticamente la lista di messaggi nel formato di conversazione previsto dal modello in uso, chiamato chat templete. La lista JSON è quindi l'interfaccia che utilizzi tu, il modello invece riceve una sequenza di token opportunamente formattata.

Perchè l'ai è come un pesce rosso?

La Memoria nelle API Stateless
Ricostruire il contesto a ogni chiamata
Le API degli LLM sono intrinsecamente 'stateless': il server non ricorda le domande precedenti. 
Questo significa, che per il server, ogni domanda è la prima domanda in assoluto.
Se chiedi: "chi è il presidente?", e subito dopo fai un'altra domanda con "quanti anni ha?" l'ai risponde con "chi?" Ha già dimenticato il soggetto.
Per mantenere il filo del discorso, dobbiamo inviare l'intera cronologia dei messaggi ad ogni chiamata.
Questo richiede una gestione intelligente del buffer locale per non eccedere il limite dei token pur mantenendo le informazioni cruciali della conversazione.
Il modello può utilizzare soltanto le informazioni disponibili nella richiesta che sta elaborando.

Come organiziamo tecnicamente questo racconto continuo?

L'Array dei Messaggi
Usiamo una struttura mista, l'array dei messaggi
- Messages List: una lista di oggetti JSON dove ogni elemento rappresenta un turno della conversazione
- Context Windows: il limite fsico che il modello puo processare in una singola chiamata (es. 128 k per GPT-4)
- Sliding Windows: tecnica per eliminare i messaggi piu vecchi quando la cronologia diventa troppo lunga
L'accumulo dei token nel contesto segue una crescita lineare rispetto al numero di messaggi scambiati
Ma cancellare i ricordi è doloroso, per fortuna esistono tecniche alternative per risparmiare spazio

- Ultimi messaggi: tieni solo gli ultimi n messaggi, perdi informazioni precedenti.
- Riassunto del contesto: invece di cancellare i vecchi messaggi, è possibile chiedere al modello di riassumere la conversazione passata e inviare solo la sintesi. Il riassunto può omettere o alterare dettagli
- Recupero da database o documenti: porti nel contesto, le informazioni pertinenti, devi selezionarle correttamente
- Token Counting: utilizzo di librerie come 'tiktoken' per calcolare preventivamente quanti token stiamo inviando ed evitare errori di overflow
- Cost Management: poichè paghiamo per ogni token inviato, una cronologia troppo vasta aumenta il costo di ogni singola risposta successiva

Ma perché il modello ha bisogno di tutto questo testo per capire?
Perché il contesto conta
Il meccanismo di self-attention permette al modello di pesare ogni parola dell'input rispetto a tutte le altre fornite nel prompt.
La qualità della risposta dipende dalla densità di informazione utile contenuta nel contesto normalizzato
inviare la cronologia non è un vezzo, è il modo per dare al modello la possibilità e gli strumenti geometrici per avere il significato della conversazione.

Definito cosa ricorda il modello possiamo definire chi è il modello 

I Tre Pilastri del Roe-Play
System, User, Assistant e Developer
Assegnare ruoli distinti permette di separare le istruzioni di controllo dai dati dell'utente, migliorando la sicurezza e la coerenza del modello.
System è la voce divina, sono le istruzioni che l'utente non può cambiare
User è l'umano che pone le domande
Assistant è il modello stesso
Developer, nelle api e nei modelli che prevedono questo ruolo, sono istruzioni dell'applicazione
Il corretto bilanciamento tra questi ruolo trasforma un modello generico in un assistente specializzato con una personalità ben definita.
Separare questi ruoli è fondamentale per la sicurezza, impedisce che un utente cerchi di ingannare un modello facendogli credere che le istruzioni siano cambiate.

Questa triade trasforma un generatore statico in un collaboratore stabile.

System Message: definiamo il comportamento 'core', le limitazioni e lo stile (es. 'sei un analista finanziario formale'). 
Condizioniamo la probabilità di uscita y, basandoci su una stringa s che rimane fissa (il system message). Forziamo il modello a pescare vocaboli da un area specifica del suo spazio latente, mettendo dei binari alla sua creatività per assicuravi che il treno arrivi alla destinazione corretta.

User Message: rappresenta l'input diretto del cliente finale o del software che interroga l'API

Assistant Message: contiene le risposte precedenti del modello, fondamentali per dare continuità logica

Il condizionamento del modello avviene attraverso la probabilità di sequenza data la stringa di sistema iniziale.

Tecniche di Steering
Come spingere questa tecnica fino a creare una vera personalità
Persona Design: è quasi psicologia applicata, puoi definire non solo il lavoro ma anche il suo umore e i suoi limiti. Creare profili psicologici o professionali dettagliati nel System Message per influenzare il lessico utilizzato dal modello.
Inserire esempi di coppie User/Assistant fittizie nella cronologia per mostrare al modello come desideeriamo che risponda (Few-Shot Prompting). Questo invece di spiegare regole a parole, si inseriscono degli esempi. Il modello è un imitatore incredibile.
Verificare la capacità del modello di rispettare vincoli negativi (es. 'non parlare mai di politica') inseriti nel ruole System (Instruction Following)

Probabilità Condizionata
Ogni parola del system message agisce come un ancora, e più il messaggio è specifico, più la distribuzione di probabilità di restringe.
E' la tecnica con cui riduciamo l'entropia della risposta prima ancora che l'utente scriva la sua prima domanda.

Determinismo e Riproducibilità
Gestire la casualità dell'inferenza.
Per applicazioni industriali, spesso vogliamo che il modello risponda nello stesso modo a parità di input. Questo contrasta con la natura stocastica degli LLM. Insomma vogliamo la riproducibilità
Ci sono parametri tecnici di 'congelare' le scelta del modello per ottenere risultati stabili e testabili.

L'architettura stessa dei transformer è pensata per essere creativa, quindi ottenere la riproducibilità non è faicile, ma abbiamo degli strumenti per domarla.

Parametri di Controllo
- Seed: valore interno che inizializza il generatore di numeri casuali, se mantenuto identico, favorisce risposte simili.
- Temperature: controlla la 'piattezza' della distribuzione di probabilità. Valori vicini a 0 rendono il modello deterministico. controlla quanto il modello può essere avventuroso. se la temperatura è 0 il modello sceglie sempre la parola più probabile
- Logprobs: possibilità di ricevere la probabilità logaritmica di ogni token per analizzare l'incertezza del modello.
- La temperatura agisce scalando i logit prima dell'applicazione della funzione softmax

Esistono altre tecniche di campionamento oltre la Temperatura

Strategie di Campionamento
- Greedy Decoding: equivale a impostare temperature=0. Il modello sceglie sempre il token con la probabilità più alta, eliminando la creatività
- Top_p (Nucleus Sampling): Alternativa alla temperatura che considera solo i token la cui probabilità cumulata raggiunge la soglia p. Invece di guardare tutti i token il modello guarda solo alla testa della distribuzione, è il modo di eliminare la coda lunga di parole.
- System Fingerprint: identificativo restituito dall'API per monitorare cambiamenti nell'infrastruttura backend che potrebbero influenzare il determinismo.

Il nostro obbiettivo finale è ridurre l'incertezza del sistema.

Incertezza Stocastica
Dall'entropia alla stabilità
Anche con il seed fisso, l'architettura distribuita del server può introdurre minime variazioni. Il determinismo assoluto è un obbiettivo asintotico.
Un'entropia bassa nella distribuzione dei token garantisce che le pipeline di produzione siano meno soggette a regressioni improvvise.
Se il codice si aspetta un JSON è il modello risponde con una poesia, il modello va in crash, usare seed e temperature = 0 è la poliza assicurativa per far si che l'ai si comporti come un modulo software prevedibile e non come un artista.

La cronologia non può crescere indefinitamente: occupa la finestra del contesto.
Il contesto del modello e, nell'esecuzione locale, anche della configurazione con cui hai caricato il modello.


In [ ]:
from openai import OpenAI

# Inizializzazione del client per la comunicazione con il server locale.
# LM Studio emula l'interfaccia di OpenAI, quindi usiamo la stessa libreria.
# Il 'base_url' punta all'indirizzo locale predefinito di LM Studio (porta 1234).
# L'API key è una stringa fissa poiché il server locale non richiede autenticazione reale.
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Definizione della memoria della conversazione.
# Questa lista di dizionari rappresenta la "storia" che passiamo al modello.
# Ogni elemento ha un 'role' (ruolo) e un 'content' (contenuto del messaggio).
context = [
    {
        "role": "system", 
        "content": "Sei un esperto di ecologia e sostenibilità. Rispondi in modo educativo e incoraggiante."
    },
    {
        "role": "user", 
        "content": "Ciao, vorrei ridurre la mia impronta di carbonio a casa."
    },
    {
        "role": "assistant", 
        "content": "Ottima iniziativa! Puoi iniziare riducendo gli sprechi energetici e migliorando l'isolamento termico."
    }
]

# Aggiungiamo un nuovo messaggio dell'utente alla cronologia esistente.
# Il modello leggerà tutto il 'context' per capire di cosa stiamo parlando.
context.append({"role": "user", "content": "Qual è il primo passo più semplice?"})

# Funzione per generare una risposta deterministica dal modello.
# Riceve in ingresso la lista dei messaggi (cronologia).
def generate_stable_response(messages_history):
    # 'completions.create' invia la richiesta al server e attende la risposta.
    return client.chat.completions.create(
        model="gpt-oss-20b", # Identificativo del modello caricato in LM Studio.
        messages=messages_history, # Passiamo l'intera cronologia dei messaggi.
        temperature=0,  # Impostato a 0 per avere risposte identiche ogni volta (niente creatività).
        seed=1234,      # Parametro opzionale per aumentare ulteriormente la riproducibilità.
        max_tokens=100  # Limite massimo di parole/token nella risposta.
    )

# Eseguiamo la prima chiamata al modello.
print("Avvio della prima generazione...")
# Estraiamo il contenuto testuale della risposta dalla struttura dati restituita (choices[0].message.content).
res1 = generate_stable_response(context).choices[0].message.content
print(f"Risposta 1: {res1}\n")

# Eseguiamo la seconda chiamata per verificare che il risultato sia lo stesso.
print("Avvio della seconda generazione per verifica determinismo...")
res2 = generate_stable_response(context).choices[0].message.content
print(f"Risposta 2: {res2}")

# Confronto tra le due risposte per assicurarci che siano identiche parola per parola.
# Questo conferma che i parametri di stabilità (temperature=0) stanno funzionando.
assert res1 == res2, "Errore: Le risposte non sono identiche!"
print("\nSuccesso: Le risposte sono identiche parola per parola.")

Avvio della prima generazione...
Risposta 1: Il primo passo più semplice (e spesso il più efficace) è **fare un “check‑up” rapido dei consumi elettrici** e poi ag

Avvio della seconda generazione per verifica determinismo...
Risposta 2: Il primo passo più semplice (e spesso il più efficace) è **fare un “check‑up” rapido dei consumi elettrici** e poi ag

Successo: Le risposte sono identiche parola per parola.
